In [1]:
import numpy as np
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib import colors
import re, os, textwrap


In [2]:
# --- helpers: color map for characters ---
CHAR_PALETTE = {
    'r': '#d62728',   # red
    'g': '#2ca02c',   # green
    'b': '#1f77b4',   # blue
    '.': '#ffffff',   # white (empty)
    'x': '#aaaaaa',   # gray (boundary / outside)
    '?': '#f0f0f0',   # fallback
}

def char_to_color(ch: str) -> str:
    return CHAR_PALETTE.get(ch, CHAR_PALETTE['?'])

# ---------- visualize full grid ----------
def visualize_grid(grid, position=None, ax=None, title=None, show=True, fontsize=12):
    """
    Visualize a 2D grid (list of lists of chars) with annotations.
    """
    rows = len(grid)
    cols = len(grid[0]) if rows > 0 else 0

    # mapping to numeric grid for colormap
    unique_chars = ['r','g','b','.','x']
    cmap_list = [char_to_color(ch) for ch in unique_chars]
    cmap = colors.ListedColormap(cmap_list)
    norm_map = {ch:i for i,ch in enumerate(unique_chars)}

    numeric_grid = np.zeros((rows, cols), dtype=int)
    for i in range(rows):
        for j in range(cols):
            numeric_grid[i, j] = norm_map.get(grid[i][j], norm_map['.'])

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(cols*0.6 + 1, rows*0.6 + 1))
        created_fig = True

    ax.imshow(numeric_grid, cmap=cmap, vmin=0, vmax=len(unique_chars)-1)
    ax.set_title(title or "Grid")
    ax.set_xticks(np.arange(-0.5, cols, 1))
    ax.set_yticks(np.arange(-0.5, rows, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(color='k', linewidth=0.5)

    # annotate characters
    for i in range(rows):
        for j in range(cols):
            ch = grid[i][j]
            ax.text(j, i, ch, ha='center', va='center', fontsize=fontsize,
                    fontweight='bold', color='black')

    # optional highlight center position
    if position is not None:
        r0, c0 = position
        if 0 <= r0 < rows and 0 <= c0 < cols:
            rect = plt.Rectangle((c0-0.5, r0-0.5), 1, 1,
                                 edgecolor='yellow', facecolor='none', lw=2)
            ax.add_patch(rect)

    if created_fig and show:
        plt.tight_layout()
        plt.show()

    return ax


In [18]:

# --- parse the log file ---
log_path = "../logs/15by15d.vis.log"  
gif_path = "../output/mis_simulation_15x15_visualizerd.gif"



with open(log_path) as f:
    lines = f.read().strip().splitlines()

rows, cols = map(int, lines[0].split())
rest = "\n".join(lines[1:])

# Extract each [[...]] block
states_raw = re.findall(r"\[\[(.*?)\]\]", rest, flags=re.DOTALL)

states = []
for s in states_raw:
    row_strings = s.split("][")
    grid = [list(row) for row in row_strings]
    states.append(grid)
duration_seconds = len(states)
# --- render each state with visualize_grid and build GIF ---
frames = []

for step_idx, grid in enumerate(states):
    fig, ax = plt.subplots(figsize=(cols*0.6 + 1, rows*0.6 + 1))
    visualize_grid(grid, ax=ax, show=False, title=f"Step {step_idx}")
    plt.tight_layout()
    fig.canvas.draw()

    # Convert canvas to RGB image array
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = img.reshape((h, w, 4))
    frames.append(img)

    plt.close(fig)


imageio.mimwrite(
    gif_path,
    frames,
    duration=300
)
print("Saved GIF to:", gif_path)


Saved GIF to: ../output/mis_simulation_15x15_visualizerd.gif
